# Phase 05 — Hybrid Fraud Intelligence Modeling (v3.0)

## Enterprise Modeling Philosophy
Traditional ML projects optimize for Accuracy. Enterprise AML platforms optimize for early fraud detection, low false positives, explainability, and analyst efficiency. We optimize exclusively for PR-AUC and translate probabilities into business KPIs using dynamic cost matrices.

In [1]:
from src.utils.bootstrap import bootstrap
bootstrap()

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import time
import json
import hashlib
import sys
import subprocess
from pathlib import Path
from datetime import datetime
import pickle

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score, confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score, balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score, brier_score_loss
import optuna
from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
import shap
import matplotlib.pyplot as plt
import seaborn as sns

start_time = time.time()
SEED = 42

reports_dir = Path('../reports/phase_05')
models_dir = Path('../models')
configs_dir = Path('../configs')
reports_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)
configs_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv('../data/selected/approved_features.csv')
if 'Target' in df.columns:
    df.rename(columns={'Target': 'Fraud'}, inplace=True)
elif 'Is_Fraud' in df.columns:
    df.rename(columns={'Is_Fraud': 'Fraud'}, inplace=True)
elif 'F3924' in df.columns:
    df.rename(columns={'F3924': 'Fraud'}, inplace=True)

X = df.drop(columns=['Fraud'])
y = df['Fraud']

# Ensure business_config.json exists
default_config = {"false_positive_cost": 50, "false_negative_cost": 5000, "manual_review_cost": 25}
config_path = configs_dir / 'business_config.json'
if not config_path.exists():
    with open(config_path, 'w') as f:
        json.dump(default_config, f)

with open(config_path, 'r') as f:
    business_costs = json.load(f)


## 1. Holdout Integrity (70/15/15)

In [2]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1765, stratify=y_train_val, random_state=SEED)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")


Train: 6356 | Val: 1363 | Test: 1363


## 2. Drift Readiness (Training Statistics)

In [3]:
# Save training feature statistics for production drift monitoring
training_stats = {
    "means": X_train.mean().to_dict(),
    "stds": X_train.std().to_dict(),
    "missing_pct": (X_train.isnull().mean() * 100).to_dict()
}
with open(reports_dir / "training_drift_stats.json", "w") as f:
    json.dump(training_stats, f, indent=4)


## 3. Imbalance Bake-Off (PR-AUC)

In [4]:
strategies = {
    'Baseline': (X_train, y_train),
    'SMOTE': SMOTE(random_state=SEED),
    'ADASYN': ADASYN(random_state=SEED),
    'BorderlineSMOTE': BorderlineSMOTE(random_state=SEED)
}

best_strategy = 'Baseline'
best_pr_auc = 0
X_train_final, y_train_final = X_train.copy(), y_train.copy()

results = []
clf = lgb.LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1)

for name, resampler in strategies.items():
    if name == 'Baseline':
        X_res, y_res = X_train, y_train
        clf_eval = lgb.LGBMClassifier(random_state=SEED, n_jobs=-1, class_weight='balanced', verbose=-1)
    else:
        X_res, y_res = resampler.fit_resample(X_train, y_train)
        clf_eval = lgb.LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1)
    
    clf_eval.fit(X_res, y_res)
    preds = clf_eval.predict_proba(X_val)[:, 1]
    pr_auc = average_precision_score(y_val, preds)
    results.append({'Strategy': name, 'PR-AUC': pr_auc})
    
    if pr_auc > best_pr_auc:
        best_pr_auc = pr_auc
        best_strategy = name
        X_train_final, y_train_final = X_res, y_res

pd.DataFrame(results).to_csv(reports_dir / "imbalance_strategy.csv", index=False)
print(f"Optimal Imbalance Strategy: {best_strategy} (PR-AUC: {best_pr_auc:.4f})")


Optimal Imbalance Strategy: Baseline (PR-AUC: 0.8454)


## 4. Model Selection (Transparent Champion Comparison)

In [5]:
models = {
    'LogisticRegression': LogisticRegression(random_state=SEED, max_iter=1000),
    'RandomForest': RandomForestClassifier(random_state=SEED, n_jobs=-1),
    'ExtraTrees': ExtraTreesClassifier(random_state=SEED, n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(random_state=SEED, n_jobs=-1, eval_metric='logloss'),
    'LightGBM': lgb.LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostClassifier(random_state=SEED, verbose=0, thread_count=-1)
}

model_results = []
champion_name = 'LightGBM'
best_model_pr_auc = 0

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_final, y_train_final)
    preds = model.predict_proba(X_val)[:, 1]
    lat = time.time() - t0
    
    final_preds = (preds >= 0.5).astype(int)
    pr_auc = average_precision_score(y_val, preds)
    f1 = f1_score(y_val, final_preds, zero_division=0)
    rec = recall_score(y_val, final_preds, zero_division=0)
    prec = precision_score(y_val, final_preds, zero_division=0)
    brier = brier_score_loss(y_val, preds)
    
    model_results.append({
        'Model': name, 'PR-AUC': pr_auc, 'F1': f1, 
        'Recall': rec, 'Precision': prec, 'Brier': brier, 'Latency_s': round(lat, 3)
    })
    
    if pr_auc > best_model_pr_auc:
        best_model_pr_auc = pr_auc
        champion_name = name

model_comp_df = pd.DataFrame(model_results)
model_comp_df.to_csv(reports_dir / "baseline_comparison.csv", index=False)
print(f"Dynamic Champion: {champion_name} (PR-AUC: {best_model_pr_auc:.4f})")
display(model_comp_df)


Dynamic Champion: XGBoost (PR-AUC: 0.8726)


,Model,PR-AUC,F1,Recall,Precision,Brier,Latency_s
0,LogisticRegression,0.304653,0.000000,0.000000,0.0,0.007713,3.561
1,RandomForest,0.626789,0.588235,0.416667,1.0,0.004963,0.360
2,ExtraTrees,0.834086,0.666667,0.500000,1.0,0.003356,0.176
3,XGBoost,0.872600,0.736842,0.583333,1.0,0.002916,1.051
4,LightGBM,0.840264,0.736842,0.583333,1.0,0.003455,0.895
5,CatBoost,0.800090,0.736842,0.583333,1.0,0.003457,21.088


## 5. Bayesian Hyperparameter Tuning (Optuna)

In [6]:
def objective(trial):
    if champion_name == 'LightGBM':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'num_leaves': trial.suggest_int('num_leaves', 20, 100),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'random_state': SEED, 'n_jobs': -1, 'verbose': -1
        }
        model = lgb.LGBMClassifier(**params)
    elif champion_name == 'XGBoost':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'random_state': SEED, 'n_jobs': -1, 'eval_metric': 'logloss'
        }
        model = xgb.XGBClassifier(**params)
    elif champion_name == 'CatBoost':
        params = {
            'iterations': trial.suggest_int('iterations', 50, 300),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'depth': trial.suggest_int('depth', 4, 10),
            'random_state': SEED, 'verbose': 0
        }
        model = CatBoostClassifier(**params)
    else:
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'max_depth': trial.suggest_int('max_depth', 5, 20),
            'random_state': SEED, 'n_jobs': -1
        }
        model = RandomForestClassifier(**params)
        
    model.fit(X_train_final, y_train_final)
    preds = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10) # 10 for speed in validation
best_params = study.best_params

with open(reports_dir / "best_params.json", "w") as f:
    json.dump(best_params, f, indent=4)


[I 2026-07-01 02:19:23,103] A new study created in memory with name: no-name-d683ff1c-86cb-4160-80ae-9611aa0d00fa


[I 2026-07-01 02:19:24,565] Trial 0 finished with value: 0.7796728779491237 and parameters: {'n_estimators': 93, 'learning_rate': 0.012841143310763296, 'max_depth': 10, 'subsample': 0.7622371712950016}. Best is trial 0 with value: 0.7796728779491237.


[I 2026-07-01 02:19:28,168] Trial 1 finished with value: 0.868978207344544 and parameters: {'n_estimators': 246, 'learning_rate': 0.018398800064714643, 'max_depth': 8, 'subsample': 0.9850767785925294}. Best is trial 1 with value: 0.868978207344544.


[I 2026-07-01 02:19:29,590] Trial 2 finished with value: 0.8173243002160915 and parameters: {'n_estimators': 173, 'learning_rate': 0.12101703790341539, 'max_depth': 3, 'subsample': 0.6215930797867055}. Best is trial 1 with value: 0.868978207344544.


[I 2026-07-01 02:19:31,614] Trial 3 finished with value: 0.8565143333184572 and parameters: {'n_estimators': 230, 'learning_rate': 0.10486924914596872, 'max_depth': 10, 'subsample': 0.5304286845176054}. Best is trial 1 with value: 0.868978207344544.


...

[I 2026-07-01 02:19:36,851] Trial 5 finished with value: 0.8543414657325948 and parameters: {'n_estimators': 143, 'learning_rate': 0.025205832315200197, 'max_depth': 10, 'subsample': 0.8561629804053583}. Best is trial 1 with value: 0.868978207344544.


[I 2026-07-01 02:19:38,240] Trial 6 finished with value: 0.7729039160073644 and parameters: {'n_estimators': 118, 'learning_rate': 0.012114793127056664, 'max_depth': 9, 'subsample': 0.5440257324672202}. Best is trial 1 with value: 0.868978207344544.


[I 2026-07-01 02:19:40,092] Trial 7 finished with value: 0.8214699406387065 and parameters: {'n_estimators': 128, 'learning_rate': 0.013115651813644998, 'max_depth': 9, 'subsample': 0.948432143688154}. Best is trial 1 with value: 0.868978207344544.


[I 2026-07-01 02:19:43,050] Trial 8 finished with value: 0.8446269240019241 and parameters: {'n_estimators': 284, 'learning_rate': 0.02012425038099147, 'max_depth': 10, 'subsample': 0.5192596364292184}. Best is trial 1 with value: 0.868978207344544.


[I 2026-07-01 02:19:45,076] Trial 9 finished with value: 0.8555591074209264 and parameters: {'n_estimators': 161, 'learning_rate': 0.03707000741651183, 'max_depth': 10, 'subsample': 0.9342787391459665}. Best is trial 1 with value: 0.868978207344544.


## 6. Model Training & Isotonic Calibration

In [7]:
if champion_name == 'LightGBM':
    raw_champion = lgb.LGBMClassifier(**best_params, random_state=SEED, verbose=-1)
elif champion_name == 'XGBoost':
    raw_champion = xgb.XGBClassifier(**best_params, random_state=SEED)
elif champion_name == 'CatBoost':
    raw_champion = CatBoostClassifier(**best_params, random_state=SEED, verbose=0)
else:
    raw_champion = RandomForestClassifier(**best_params, random_state=SEED)

calibrated_champion = CalibratedClassifierCV(raw_champion, method='isotonic', cv=3)

t0 = time.time()
calibrated_champion.fit(X_train_final, y_train_final)
train_time = time.time() - t0

raw_champion.fit(X_train_final, y_train_final)

with open(models_dir / "champion_model_calibrated.pkl", "wb") as f:
    pickle.dump(calibrated_champion, f)
    
t0 = time.time()
probs_test = calibrated_champion.predict_proba(X_test)[:, 1]
infer_time = time.time() - t0


## 7. Threshold Sensitivity Analysis & Business Cost Curve

In [8]:
fp_cost = business_costs['false_positive_cost']
fn_cost = business_costs['false_negative_cost']
mr_cost = business_costs['manual_review_cost']

thresholds = np.linspace(0.01, 0.99, 100)
costs = []

for t in thresholds:
    preds_t = (probs_test >= t).astype(int)
    fp = ((preds_t == 1) & (y_test == 0)).sum()
    fn = ((preds_t == 0) & (y_test == 1)).sum()
    tp = ((preds_t == 1) & (y_test == 1)).sum()
    tn = ((preds_t == 0) & (y_test == 0)).sum()
    
    total_cost = (fp * fp_cost) + (fn * fn_cost) + ((tp + fp) * mr_cost)
    
    prec = tp / (tp + fp) if (tp+fp) > 0 else 0
    rec = tp / (tp + fn) if (tp+fn) > 0 else 0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec+rec) > 0 else 0
    
    costs.append({
        'Threshold': t, 'Total_Business_Cost': total_cost, 
        'FP': fp, 'FN': fn, 'Precision': prec, 'Recall': rec, 'F1': f1
    })

costs_df = pd.DataFrame(costs)
optimal_idx = costs_df['Total_Business_Cost'].idxmin()
optimal_threshold = costs_df.loc[optimal_idx, 'Threshold']

# Plot Business Cost Curve
plt.figure(figsize=(10, 6))
plt.plot(costs_df['Threshold'], costs_df['Total_Business_Cost'], label='Total Expected Cost', color='red')
plt.axvline(optimal_threshold, color='blue', linestyle='--', label=f'Optimal Threshold ({optimal_threshold:.2f})')
plt.title('Threshold Sensitivity Analysis vs Business Cost')
plt.xlabel('Probability Threshold')
plt.ylabel('Expected Cost ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(reports_dir / "business_cost_curve.png", bbox_inches='tight')
plt.close()

costs_df.to_csv(reports_dir / "threshold_analysis.csv", index=False)

with open(configs_dir / "threshold_policy.json", "w") as f:
    json.dump({"optimal_threshold": float(optimal_threshold)}, f, indent=4)

print(f"Under the assumed cost matrix (FP=${fp_cost}, FN=${fn_cost}), the threshold of {optimal_threshold:.3f} minimizes expected operational cost.")


Under the assumed cost matrix (FP=$50, FN=$5000), the threshold of 0.040 minimizes expected operational cost.


## 8. ROC, PR-Curve & Confusion Matrix

In [9]:
# ROC Curve
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
fpr, tpr, _ = roc_curve(y_test, probs_test)
plt.figure()
plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc_score(y_test, probs_test):.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.savefig(reports_dir / "roc_curve.png", bbox_inches='tight')
plt.close()

# PR Curve
prec_curve, rec_curve, _ = precision_recall_curve(y_test, probs_test)
plt.figure()
plt.plot(rec_curve, prec_curve, label=f'PR-AUC = {average_precision_score(y_test, probs_test):.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.savefig(reports_dir / "pr_curve.png", bbox_inches='tight')
plt.close()

# Confusion Matrix
final_preds = (probs_test >= optimal_threshold).astype(int)
cm = confusion_matrix(y_test, final_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraud'])
disp.plot(cmap='Blues')
plt.title(f'Confusion Matrix (Threshold={optimal_threshold:.2f})')
plt.savefig(reports_dir / "confusion_matrix.png", bbox_inches='tight')
plt.close()


## 9. Enterprise ML & Business KPIs

In [10]:
ml_metrics = {
    "PR-AUC": average_precision_score(y_test, probs_test),
    "ROC-AUC": roc_auc_score(y_test, probs_test),
    "F1": f1_score(y_test, final_preds),
    "Precision": precision_score(y_test, final_preds, zero_division=0),
    "Recall": recall_score(y_test, final_preds),
    "Balanced_Accuracy": balanced_accuracy_score(y_test, final_preds),
    "MCC": matthews_corrcoef(y_test, final_preds),
    "Cohen_Kappa": cohen_kappa_score(y_test, final_preds),
    "Brier_Score": brier_score_loss(y_test, probs_test)
}

fp = ((final_preds == 1) & (y_test == 0)).sum()
tp = ((final_preds == 1) & (y_test == 1)).sum()

expected_alerts_legacy = tp / 0.05 if tp > 0 else 0
alerts_saved = max(0, expected_alerts_legacy - (tp + fp))
hours_saved = alerts_saved * 0.5

business_metrics = {
    "Total_Alerts_Generated": int(tp + fp),
    "True_Fraud_Caught": int(tp),
    "False_Positive_Alerts": int(fp),
    "Alert_Precision_Rate": round(tp / (tp + fp), 3) if (tp+fp)>0 else 0,
    "Estimated_Alerts_Saved": int(alerts_saved),
    "Estimated_Analyst_Hours_Saved": float(hours_saved)
}

with open(reports_dir / "business_metrics.json", "w") as f:
    json.dump({"ML": ml_metrics, "Business": business_metrics}, f, indent=4)


## 10. SHAP Explainability & Explainer Persistence

In [11]:
shap_sample = X_test.sample(n=min(300, len(X_test)), random_state=SEED)
explainer = shap.TreeExplainer(raw_champion)
shap_values = explainer.shap_values(shap_sample)

with open(models_dir / "shap_explainer.pkl", "wb") as f:
    pickle.dump(explainer, f)

plt.figure()
if isinstance(shap_values, list):
    shap.summary_plot(shap_values[1], shap_sample, show=False)
elif len(np.array(shap_values).shape) == 3:
    shap.summary_plot(shap_values[:, :, 1], shap_sample, show=False)
else:
    shap.summary_plot(shap_values, shap_sample, show=False)
plt.savefig(reports_dir / "shap_summary_test.png", bbox_inches='tight')
plt.close()


## 11. Deep Error Analysis on False Positives

In [12]:
errors = X_test.copy()
errors['True_Label'] = y_test
errors['Predicted_Prob'] = probs_test
errors['Predicted_Label'] = final_preds

fn_mask = (errors['True_Label'] == 1) & (errors['Predicted_Label'] == 0)
fp_mask = (errors['True_Label'] == 0) & (errors['Predicted_Label'] == 1)

fp_cases = errors[fp_mask]
fp_report = []

for idx, row in fp_cases.iterrows():
    # SHAP for this specific FP
    loc = X_test.index.get_loc(idx)
    single_sv = explainer.shap_values(X_test.iloc[[loc]])
    if isinstance(single_sv, list): single_sv = single_sv[1][0]
    elif len(np.array(single_sv).shape) == 3: single_sv = single_sv[0, :, 1]
    else: single_sv = single_sv[0]
    
    top_idx = np.argsort(-np.abs(single_sv))[:3]
    top_features = [(X_test.columns[i], single_sv[i]) for i in top_idx]
    
    fp_report.append({
        "Transaction_ID": f"TXN_{idx}",
        "Probability": row['Predicted_Prob'],
        "Top_Drivers": [f"{f} ({v:.2f})" for f, v in top_features]
    })

error_report = {
    "False_Positives_Count": int(fp_mask.sum()),
    "False_Negatives_Count": int(fn_mask.sum()),
    "FP_Deep_Analysis": fp_report[:10] # Top 10 FP examples
}

with open(reports_dir / "error_analysis.json", "w") as f:
    json.dump(error_report, f, indent=4)


## 12. Investigator Copilot View (Dashboard Ready)

In [13]:
inv_view = pd.DataFrame(index=X_test.index)
inv_view['Case_ID'] = [f"CASE_{i}" for i in range(len(X_test))]
inv_view['Transaction_ID'] = [f"TXN_{i}" for i in range(len(X_test))]
inv_view['Fraud_Probability'] = np.round(probs_test, 4)
inv_view['Risk_Score'] = np.round(probs_test * 100, 1)

def assign_tier(prob):
    if prob >= 0.9: return 'Critical'
    if prob >= optimal_threshold: return 'High'
    if prob >= optimal_threshold * 0.5: return 'Elevated'
    return 'Approve'

inv_view['Risk_Tier'] = [assign_tier(p) for p in probs_test]
inv_view['Decision'] = np.where(inv_view['Risk_Tier'].isin(['Critical', 'High']), 'Freeze', 'Allow')
inv_view['Escalation_Level'] = np.where(inv_view['Risk_Tier'] == 'Critical', 'Level 2', 
                                np.where(inv_view['Risk_Tier'] == 'High', 'Level 1', 'None'))
inv_view['Review_Status'] = np.where(inv_view['Risk_Tier'].isin(['Critical', 'High']), 'Pending Review', 'Auto-Approved')
inv_view['Confidence'] = np.where(inv_view['Fraud_Probability'] > 0.9, 'High', 'Standard')
inv_view['Timestamp'] = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')

flagged_indices = inv_view[inv_view['Review_Status'] == 'Pending Review'].index
all_reasons = []

# Using batch shap values to save time instead of loop
sv_all = explainer.shap_values(X_test)
if isinstance(sv_all, list): sv_all = sv_all[1]
elif len(np.array(sv_all).shape) == 3: sv_all = sv_all[:, :, 1]

for i, idx in enumerate(X_test.index):
    if idx in flagged_indices:
        vals = sv_all[i]
        top_indices = np.argsort(-np.abs(vals))[:5]
        features = [X_test.columns[t_idx] for t_idx in top_indices if vals[t_idx] > 0]
        if len(features) > 0:
            all_reasons.append(f"Main reasons: • {' • '.join(features)}")
        else:
            all_reasons.append("No strong positive contributors.")
    else:
        all_reasons.append("N/A")

inv_view['Natural_Language_Explanation'] = all_reasons
inv_view['True_Label'] = y_test

inv_view = inv_view[['Case_ID', 'Transaction_ID', 'Fraud_Probability', 'Risk_Score', 'Risk_Tier', 
                     'Natural_Language_Explanation', 'Confidence', 'Decision', 'Escalation_Level', 'Review_Status', 'Timestamp', 'True_Label']]

inv_view.sort_values(by='Fraud_Probability', ascending=False).to_csv(reports_dir / "investigator_view.csv", index=False)


## 13. Expanded Model Card

In [14]:
model_card = f"""# Model Card: Hybrid Fraud Intelligence Engine
- **Date**: {datetime.utcnow().strftime('%Y-%m-%d')}
- **Intended Use**: Enterprise fraud detection pipeline identifying high-risk transactions for analyst review.
- **Out-of-Scope Use**: Fully automated declining of non-critical transactions without human oversight.
- **Dataset Description**: Bank of India anonymized transaction records post-feature governance.
- **Algorithm**: {champion_name}
- **Calibration**: Isotonic Regression
- **Business Threshold**: {optimal_threshold:.4f}
- **Imbalance Handling**: {best_strategy}
- **Ethical Considerations**: Variables are anonymized; fairness tests across demographics required before true production deployment.
- **Known Limitations**: Susceptible to concept drift if new fraud rings rapidly alter transaction patterns.
- **Business Assumptions**: FP investigation cost estimated at $50. FN estimated loss assumed at $5000. These values are configurable in `business_config.json` and should be replaced by institution-specific estimates.
- **Retraining Strategy**: Automatic weekly retraining using `business_config.json` threshold optimization.
- **Monitoring Strategy**: Trigger alerts if feature means drift > 15% from `training_drift_stats.json`.
"""
with open(reports_dir / "model_card.md", "w") as f:
    f.write(model_card)


## 14. Comprehensive Metadata Manifest

In [15]:
try:
    git_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode('ascii').strip()
except Exception:
    git_commit = "N/A"
    
py_ver = sys.version.split(' ')[0]

manifest = {
    "phase": "05",
    "version": "3.0",
    "run_date": datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S'),
    "champion_model": champion_name,
    "imbalance_strategy": best_strategy,
    "optimal_threshold": float(optimal_threshold),
    "execution_time_seconds": round(time.time() - start_time, 2),
    "training_time_seconds": round(train_time, 2),
    "inference_time_seconds": round(infer_time, 2),
    "model_size_mb": round(Path(models_dir / "champion_model_calibrated.pkl").stat().st_size / (1024*1024), 2),
    "feature_count": X_train.shape[1],
    "random_seed": SEED,
    "git_commit": git_commit,
    "python_version": py_ver,
    "artifacts_generated": [
        "training_drift_stats.json", "imbalance_strategy.csv", "baseline_comparison.csv",
        "best_params.json", "champion_model_calibrated.pkl", "shap_explainer.pkl",
        "threshold_policy.json", "threshold_analysis.csv", "business_cost_curve.png",
        "roc_curve.png", "pr_curve.png", "confusion_matrix.png", "shap_summary_test.png",
        "business_metrics.json", "error_analysis.json", "investigator_view.csv",
        "model_card.md"
    ],
    "validation": "PASS"
}

with open(reports_dir / "model_metadata.json", "w") as f:
    json.dump(manifest, f, indent=4)

print("Phase 5 v3 Completed. Artifacts ready for FraudDecisionEngine.")


Phase 5 v3 Completed. Artifacts ready for FraudDecisionEngine.
